In [ ]:
%pip install -U huggingface_hub

In [ ]:
from huggingface_hub import login

login("hf_aaaaaaaaaaaaaaaaaaaaaaaaaaaaa")

In [ ]:
from huggingface_hub import create_repo
from huggingface_hub import delete_repo

create_repo(
    "jayanta2025/flickr31k-images",
    repo_type="dataset",
    exist_ok=True,
)

In [ ]:
import os
from collections import defaultdict

image_dir = "../../Flicker 30k hf/Images"  # change this each run
folders = os.listdir(image_dir)
folders

In [ ]:
image_folders = []
for folder in folders:
    if 'demo' in folder:
        continue

    chunks = os.listdir(os.path.join(image_dir, folder))
    
    for chunk in chunks:
        image_folders.append(os.path.join(image_dir, folder, chunk))

In [ ]:
print(len(image_folders))
# image_folders

delete_repo(
    repo_id="jayanta2025/flickr31k-images",
    repo_type="dataset",
)

In [ ]:
image_folders[-2:]

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

for folder in image_folders[-1:]:
    print("Uploading from...", folder.split('/')[-1])
    api.upload_folder(
        folder_path=folder,
        path_in_repo="Images4",
        repo_id="jayanta2025/flickr31k-images",
        repo_type="dataset",
    )
    print("[DONE]", folder.split('/')[-1])

### Remove duplicate from HF

In [ ]:
# for folder in image_folders:
#     print(folder, len(os.listdir(folder)))

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "jayanta2025/flickr31k-images"
repo_type = "dataset"

# Get all files in the repo
repo_files = set(api.list_repo_files(repo_id=repo_id, repo_type=repo_type))

In [ ]:
print(len(repo_files)) # expected 31783

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "jayanta2025/flickr31k-images"
repo_type = "dataset"

folder1 = "Images"   # Delete from here if duplicate
folder2 = "Images2"   # Keep this one
folder3 = "Images3"   # Keep this one
folder4 = "Images4"   # Keep this one

upload = []
delete = []

print(f"[CHECKING]...")
for i in range(len(image_folders)):
    chunk = image_folders[i]
    files = os.listdir(chunk)

    cnt = 0
    not_in = 0
    not_uploaded = 0
    for file in files:
        image_name = file
        path1 = f"{folder1}/{image_name}"
        path2 = f"{folder2}/{image_name}"
        path3 = f"{folder3}/{image_name}"
        path4 = f"{folder4}/{image_name}"

        if path3 in repo_files and path4 in repo_files:
            delete.append((path3, image_name))
            cnt += 1

            # print(f"Duplicate found!\nDeleting {path1}")
            # api.delete_file(
            #     repo_id=repo_id,
            #     repo_type=repo_type,
            #     path_in_repo=path1,
            #     commit_message=f"Remove duplicate {image_name} from {folder1}",
            # )

            # print("Deleted successfully.")

        if path1 not in repo_files and path2 not in repo_files and path3 not in repo_files and path4 not in repo_files:
            upload.append((chunk, image_name))
            not_uploaded += 1

    if cnt > 0 or not_in > 0 or not_uploaded>0:
        print("[DEBUG]", chunk, len(files))
        
    if not_uploaded > 0:
        print(f"[UPLOAD] Upload {not_uploaded} images")
    if cnt > 0:
        print(f"[DUPLICATES] Found {cnt} duplicates.")
    if not_in > 0:
        print(f"[COMPARE] {not_in} neither in {folder1} nor in {folder2}")



In [ ]:
len(delete)

In [ ]:
from huggingface_hub import CommitOperationDelete

operations = []

for path, image in  delete:
    operations.append(CommitOperationDelete(path_in_repo=path))

In [ ]:
len(operations)

In [ ]:
# api.create_commit(
#     repo_id="jayanta2025/flickr31k-images",
#     repo_type="dataset",
#     operations=operations,
#     commit_message="Remove duplicate images",
# )

### Mongo-DB Update, Append Image_url

In [ ]:
import os
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()

# Config
DB_NAME = "database_embd"
COLLECTION_NAME = "embeddings_31k"
MONGO_URI = os.getenv('MONGODB_DRIVER_STRING')

EMBEDDING_PATH = "../generate_embedding/embeddings.pkl"
CAPION_PATH = "../../Flicker 30k hf/captions.txt"

# Connect to MongoDB
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    print("✅ Connected to MongoDB successfully")
    
except Exception as e:
    print("❌ Connection failed:", e)

collection = client[DB_NAME][COLLECTION_NAME]
print(f"\nCurrent DB: '{DB_NAME}', Collection: '{COLLECTION_NAME}'")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

repo_id = "jayanta2025/flickr31k-images"
repo_type = "dataset"

# Get all files in the repo
files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)

In [ ]:
print(len(files), len(set(files)))

In [ ]:
image_map = {}
for file in files[1:]:
    folder, image = file.split('/')
    image_map[image] = file

In [ ]:
# 1000092795.jpg': 'Images/1000092795.jpg',
#  '10002456.jpg': 'Images/10002456.jpg',
#  '1000268201.jpg': 'Images/1000268201.jpg',
#  '1000344755.jpg': 'Images/1000344755.jpg',
#  '1000366164.jpg': 'Images/1000366164.jpg',
#  '1000523639.jpg': 'Images/1000523639.jpg',
#  '1000919630.jpg': 'Images/1000919630.jpg',


In [ ]:
image_id = '1000092795.jpg'
print(image_id, type(image_id))
res = collection.find_one({"image_id": image_id})
for key, val in res.items():
    print(key)

In [ ]:
image_id = '1000092795.jpg'
image_url = image_map[image_id]
print(image_id, image_url)

In [ ]:
collection.update_one({"image_id": image_id},
            {"$set": {"image_url": image_url}})

In [ ]:
image_id = '1000092795.jpg'
print(image_id, type(image_id))
res = collection.find_one({"image_id": image_id})
for key, val in res.items():
    print(key)

In [ ]:
from pymongo import UpdateOne

operations = []

for image_id, image_url in image_map.items():
    operations.append(
        UpdateOne(
            {"image_id": image_id},
            {"$set": {"image_url": image_url}}
        )
    )

    if len(operations) == 1000:
        result = collection.bulk_write(operations)
        print(result.bulk_api_result)
        operations = []
        
if operations:
        result = collection.bulk_write(operations)
        print(result.bulk_api_result)